In [1]:
"""
2025-08-29-Make a dilution series in a BioER Deepwell plate

Deck
----
TIP_CAR_480_A00 @ rails 25
  [0] 1000 µL CO-RE HFT (filtered)  – tips_00   
  [1] 50 µL CO-RE filtered         – tips_01   
  [2] 10 µL CO-RE filtered 

MFX_CAR_L5_base @ rails 19
  pos-0  BioER Plate
  

MFX_CAR_L5_base @ rails 13
  pos-1  MFX_DWP_module_188042 → AGenBio_1_troughplate_100000uL_Fl (water)
  pos-0  position of PCR plate in other scripts


MFX_CAR_L5_base @ rails 7      (reserved / empty)
  pos-0  MFX_DWP_module_188042 → Tube rack (24-well, 1.5 mL, OT-2)
    ▸ D1 contains 1 µM dsDNA (source)
    NOT D6; CHECK yo'self

Channels
--------
2 → water load & fill           (1000 µL tip, tips_00)
3 → serial dilution transfers   (15 × 1000 µL tips, tips_00)

Author : Harley King

Next Steps: Currently uses tips from cols A2, A3. Next to make it variable e.g. A5, A6.
"""


"\n2025-08-29-Make a dilution series in a BioER Deepwell plate\n\nDeck\n----\nTIP_CAR_480_A00 @ rails 25\n  [0] 1000 µL CO-RE HFT (filtered)  – tips_00   \n  [1] 50 µL CO-RE filtered         – tips_01   \n  [2] 10 µL CO-RE filtered \n\nMFX_CAR_L5_base @ rails 19\n  pos-0  BioER Plate\n\n\nMFX_CAR_L5_base @ rails 13\n  pos-1  MFX_DWP_module_188042 → AGenBio_1_troughplate_100000uL_Fl (water)\n  pos-0  position of PCR plate in other scripts\n\n\nMFX_CAR_L5_base @ rails 7      (reserved / empty)\n  pos-0  MFX_DWP_module_188042 → Tube rack (24-well, 1.5 mL, OT-2)\n    ▸ D1 contains 1 µM dsDNA (source)\n    NOT D6; CHECK yo'self\n\nChannels\n--------\n2 → water load & fill           (1000 µL tip, tips_00)\n3 → serial dilution transfers   (15 × 1000 µL tips, tips_00)\n\nAuthor : Harley King\n\nNext Steps: Currently uses tips from cols A2, A3. Next to make it variable e.g. A5, A6.\n"

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import asyncio
from typing import List, Iterator

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import STARLetDeck, MFX_CAR_L5_base, TIP_CAR_480_A00
# from pylabrobot.resources.hamilton.mfx_modules import MFX_DWP_module_188042
from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped
from pylabrobot.resources.opentrons.tube_racks import (
    opentrons_24_tuberack_generic_1point5ml_snapcap_short,
)
from pylabrobot.resources.tube_adapter import TubeRackAdapter
from pylabrobot.resources.agenbio.plates import AGenBio_1_troughplate_100000uL_Fl
from pylabrobot.resources.bioer.plates import BioER_96_wellplate_Vb_2200ul
from pylabrobot.resources import (
    LTF, #10ul filtered
    TIP_50ul_w_filter, # 50 µL filtered
    HTF     # 1000 µL filtered 
)

###############################################################################
# User-adjustable parameters
###############################################################################

START_TIP = "A1"          # first 1000 µL tip to pick up (row B, col 4)
MIX_VOL   = 800           # µL, mixing volume
MIX_CYC   = 3             # cycles per tube
ASP_RATE  = None          # µL s-1, None = PLR default
DSP_RATE  = None
CHANNEL_WATER   = 1       # index = channel-2 (0-based)
CHANNEL_DILUTE  = 1       # index = channel-3 (0-based)

###############################################################################
# 0) build LiquidHandler + deck
###############################################################################
backend = STARBackend()
lh      = LiquidHandler(backend=backend, deck=STARLetDeck())
await lh.setup(skip_autoload=True)

###############################################################################
# 1) carriers, modules & labware
###############################################################################
# --- tip carrier -------------------------------------------------------------
# --- tip carrier -------------------------------------------------------------
tip_car = TIP_CAR_480_A00("tip_car")
lh.deck.assign_child_resource(tip_car, rails=25)

tiprack_1000 = HTF("tips_00")              # 1000 µL filter tips (slot-0)
tiprack_50   = TIP_50ul_w_filter("tips_01") #  50 µL filter tips (slot-1)

# mount the racks
tip_car[0] = tiprack_1000          # OR:  tip_car[0].assign_child_resource(tiprack_1000)
tip_car[1] = tiprack_50


# --- carrier @ rail-19  ------------------------------------------------------
dwp_mod_dest   = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_dest")
car_19 = MFX_CAR_L5_base(
    "car_19",
    modules={
        0: dwp_mod_dest
    }
)
lh.deck.assign_child_resource(car_19, rails=19)
dw_plate = BioER_96_wellplate_Vb_2200ul('dw_plate')
dwp_mod_dest.assign_child_resource(dw_plate)

# # labware
# tuberack_dest = opentrons_24_tuberack_generic_1point5ml_snapcap_short("dest_rack")
# dest_offset_x = (127.76 - tuberack_dest._size_x) / 2
# dest_offset_y = (85.48  - tuberack_dest._size_y) / 2

# adapter_dest = TubeRackAdapter(
#     name="dest_rack_adapter",
#     size_x=127.76,
#     size_y=85.48,
#     size_z=tuberack_dest._size_z,              # external height of the frame
#     model="tube_rack_adapter",
#     dx=dest_offset_x,
#     dy=dest_offset_y,
#     dz=0,
#     adapter_hole_size_x=tuberack_dest._size_x,
#     adapter_hole_size_y=tuberack_dest._size_y,
#     adapter_hole_size_z=tuberack_dest._size_z
# )
# adapter_dest.assign_child_resource(tuberack_dest)
# dwp_mod_dest.assign_child_resource(adapter_dest)

# ----------carrier @ rail 13: water trough-------------------
# water reservoir
dwp_mod_trough = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_trough")
car_13 = MFX_CAR_L5_base(
    "car_13",
    modules={
        1: dwp_mod_trough, # want in this position so I can add a plate to [0] position for qPCR
    }
)
lh.deck.assign_child_resource(car_13, rails=13)
water_plate = AGenBio_1_troughplate_100000uL_Fl("water_plate")
dwp_mod_trough.assign_child_resource(water_plate)

# --- carrier @ rail-7: 2mL tube with dsDNA ----------------------------
dwp_mod_src = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_src")
car_07 = MFX_CAR_L5_base(
    "car_07",
    modules={
        0: dwp_mod_src,
    }
)
lh.deck.assign_child_resource(car_07, rails=7)

tuberack_src = opentrons_24_tuberack_generic_1point5ml_snapcap_short("src_rack")
src_offset_x = (127.76 - tuberack_src._size_x) / 2
src_offset_y = (85.48  - tuberack_src._size_y) / 2

adapter_src = TubeRackAdapter(
    name="src_rack_adapter",
    size_x=127.76,
    size_y=85.48,
    size_z=tuberack_src._size_z,              # external height of the frame
    model="tube_rack_adapter",
    dx=src_offset_x,
    dy=src_offset_y,
    dz=0,
    adapter_hole_size_x=tuberack_src._size_x,
    adapter_hole_size_y=tuberack_src._size_y,
    adapter_hole_size_z=tuberack_src._size_z
)
adapter_src.assign_child_resource(tuberack_src)
dwp_mod_src.assign_child_resource(adapter_src)

In [4]:
ROWS = "ABCDEFGH"
CHANNELS_8 = list(range(8))
TIPSTART = "A4:H4" # from which col pickup tips?


move_water = dict(
    lld_mode=[STARBackend.LLDMode.GAMMA]*8,
    gamma_lld_sensitivity=[2]*8,          # tweak if needed
    immersion_depth=[1]*8,
    immersion_depth_direction=[0]*8,
    surface_following_distance=[1]*8,
    transport_air_volume=[0]*8,
    settling_time=[1]*8
)

async def preload_water():
    await lh.pick_up_tips(tiprack_1000[TIPSTART], use_channels=CHANNELS_8)  # HTF = 1000 µL filtered
    
    for col in range(1, 7, 5): # -> 1, 6 
        # aspirate water
        await lh.aspirate(
                water_plate["A1"]*8, 
                vols=[1000]*8, 
                use_channels=CHANNELS_8,
                **move_water)
        # quick dispense back into plate    
        await lh.dispense(water_plate["A1"]*8, vols=[50]*8, use_channels=CHANNELS_8, **move_water)
        # dispense into deepwell column,e.g. A1..H1
        
        dest_col = dw_plate[f"A{col}:H{col}"]
        await lh.dispense(
                dest_col,
                vols=[900]*8,
                use_channels=CHANNELS_8,
                liquid_height=[20]*8, 
                settling_time=[1]*8
            )
        
        await lh.dispense(water_plate["A1"]*8, vols=[50]*8, use_channels=CHANNELS_8, **move_water, blow_out=[1]*8)
    
    await lh.drop_tips(tiprack_1000[TIPSTART])  # to default waste


ROWS = "ABCDEFGH"
CHANNEL_DILUTE = 4  # single channel index (0-7)

def iter_tips_from_A2_H2_then_A3_H3():
    """Yield A2..H2 then A3..H3 for single-tip pickups."""
    for rownum in ("2", "3"):
        for letter in ROWS:
            yield f"{letter}{rownum}"

async def _pick_next_tip(tip_iter):
    try:
        tip_pos = next(tip_iter)  # e.g., "A2"
    except StopIteration:
        raise RuntimeError("Not enough tips: need up to 15 single tips (A2:H2 then A3:H3).")
    await lh.pick_up_tips(tiprack_1000[tip_pos], use_channels=[CHANNEL_DILUTE])

async def _dispense_with_mix(dest_well):
    # Mix into preloaded 900 µL water (assumes you've run preload_water() first)
    await lh.dispense(
        dest_well,
        vols=[100],
        use_channels=[CHANNEL_DILUTE],
        lld_mode=[STARBackend.LLDMode.GAMMA],
        mix_volume=[900],                 # safe because wells contain ~900 µL water already
        mix_cycles=[2],
        mix_speed=[400],
        mix_surface_following_distance=[15],
        blow_out=[1]
    )

async def serial_dilution():
    tips = iter_tips_from_A2_H2_then_A3_H3()

    # 1) Seed: tube -> A1
    await _pick_next_tip(tips)
    try:
        await lh.aspirate(
            tuberack_src["D1"],
            vols=[100],
            use_channels=[CHANNEL_DILUTE],
            lld_mode=[STARBackend.LLDMode.GAMMA],
            immersion_depth=[2],
            mix_volume=[100],
            mix_cycles=[2],
            mix_position_from_liquid_surface=[1],
            mix_surface_following_distance=[2],
            surface_following_distance=[2]
        )
    except Exception:
        # fallback if LLD fails on a low tube
        await lh.aspirate(
            tuberack_src["D1"],
            vols=[100],
            use_channels=[CHANNEL_DILUTE],
            liquid_height=[1]
        )
    await _dispense_with_mix(dw_plate["A1"])
    await lh.discard_tips()

    # 2) Column 1 serial: A1->B1->...->H1 (7 hops)
    for i in range(len(ROWS) - 1):  # 0..6 -> A..G as sources
        src = f"{ROWS[i]}1"
        dst = f"{ROWS[i+1]}1"
        await _pick_next_tip(tips)
        await lh.aspirate(
            dw_plate[src],
            vols=[100],
            use_channels=[CHANNEL_DILUTE],
            lld_mode=[STARBackend.LLDMode.GAMMA],
            immersion_depth=[2], 
            mix_volume=[100],
            mix_cycles=[2],
            mix_position_from_liquid_surface=[1],
            mix_surface_following_distance=[2],
        )
        await _dispense_with_mix(dw_plate[dst])
        await lh.discard_tips()

    # 3) Bridge: H1 -> A6
    await _pick_next_tip(tips)
    await lh.aspirate(
        dw_plate["H1"],
        vols=[100],
        use_channels=[CHANNEL_DILUTE],
        lld_mode=[STARBackend.LLDMode.GAMMA],
        immersion_depth=[2]
    )
    await _dispense_with_mix(dw_plate["A6"])
    await lh.discard_tips()

    # 4) Column 6 serial: A6->B6->...->G6 (STOP before H6, which remains blank)
    for i in range(len(ROWS) - 2):  # 0..5 -> A..F as sources, last dest is G
        src = f"{ROWS[i]}6"
        dst = f"{ROWS[i+1]}6"
        await _pick_next_tip(tips)
        await lh.aspirate(
            dw_plate[src],
            vols=[100],
            use_channels=[CHANNEL_DILUTE],
            lld_mode=[STARBackend.LLDMode.GAMMA],
            immersion_depth=[2]
        )
        await _dispense_with_mix(dw_plate[dst])
        await lh.discard_tips()






In [ ]:
await preload_water()
await serial_dilution()

In [ ]:
# def get_tipstart_pos (start: str):
        # Return the left side (start well) from a range like 'A2:H2'.
        # If there's no ':', returns the input trimmed.
    # return start.s.strip().split(':', 1)[0].strip()

# CHANNEL_DILUTE = 4
# SINGLE_TIP = f"{get_tipstart_pos(TIPSTART)}" # e.g. "A2"
# async def serial_dilution():
    

#   # for well in 
#     await lh.pick_up_tips(tiprack_1000[SINGLE_TIP], use_channels=[CHANNEL_DILUTE])
#     # first transfer: dsDNA D1 → dp_plate[A1]
#     try:
#         await lh.aspirate(
#             tuberack_src["D1"],
#             vols=[100],
#             use_channels=[CHANNEL_DILUTE],
#             lld_mode=[STARBackend.LLDMode.GAMMA],
#             immersion_depth=[1],
#             mix_volume=[100],
#             mix_cycles=[2],
#             mix_position_from_liquid_surface = [1],
#             mix_speed = [400],
#             mix_surface_following_distance=[2]
#         )
#     except: #if fluid is low, LLDMode.Gamma throws an error
#         await lh.aspirate(
#             tuberack_src["D1"],
#             vols=[100],
#             use_channels=[CHANNEL_DILUTE],
#             liquid_height=[1]
#         )
        
#     await lh.dispense(
#         dw_plate["A1"],
#         vols=[100],
#         use_channels=[CHANNEL_DILUTE],
#         mix_volume=[900],
#         mix_cycles=[2],
#         mix_speed=[400],
#         mix_surface_following_distance=[20],
#         lld_mode=[STARBackend.LLDMode.GAMMA],
#         blow_out=[1]
#     )
    
#     await lh.discard_tips()
#     for cycle in range(1, 7, 5):
#         for i, row in enumerate(ROWS):
#             await lh.pick_up_tips(tiprack_1000["B2"], use_channels=[CHANNEL_DILUTE])
#             src_well = dw_plate["A1"]
#             dest_well = dw_plate["B1"]

#         await lh.aspirate(
#             src_well,
#             vols=[100],
#             liquid_height=[20],
#             use_channels=[CHANNEL_DILUTE],
#             lld_mode=[STARBackend.LLDMode.GAMMA],
#             immersion_depth=[3]
#         )
#         await lh.dispense(
#             dest_well,
#             vols=[100],
#             use_channels=[CHANNEL_DILUTE],
#             mix_volume=[900],
#             mix_cycles=[2],
#             mix_speed=[400],
#             mix_surface_following_distance=[20],
#             lld_mode=[STARBackend.LLDMode.GAMMA],
#             blow_out=[1]
#         )
#         await lh.discard_tips()
#         # next tip e.g. B1

# ###############################################################################
# # 5) run protocol
# ###############################################################################

# await preload_water()
# await serial_dilution()

In [7]:
# await lh.dispense(water_plate["A1"]*8, vols=[10]*8, liquid_height=[2]*8, use_channels=CHANNELS_8)
# await lh.drop_tips(tiprack_1000["B2"], use_channels=[CHANNEL_DILUTE])
# await lh.discard_tips()
await lh.stop()